# Spectromètre : intensité en fonction de la longueur d'onde

In [2]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

plt.rcParams['figure.dpi'] = 150

## Paramètres

In [3]:
# ---------------- Paramètres du montage (à définir) ----------------
lbd = 550e-9      # longueur d'onde centrale (m)
f1 = None         # focale de la 1ère lentille (m)
f2 = None         # focale de la 2e lentille (m)      -> requis pour la calibration théorique
L = None          # taille de l'ouverture (m)         -> sert à estimer la résolution
Lambda = None     # pas du réseau (m)                 -> requis pour la calibration théorique
ordre = 1         # ordre de diffraction utilisé
theta_i = 0.0     # angle d'incidence sur le réseau (rad)
pixel = 1.55e-6   # pas des pixels de l'IMX477 (m)
sens = 1          # +1 si lbd augmente avec l'indice de pixel, -1 sinon (image inversée)

# ---------------- Analyse ----------------
dossier_mesure = "acquisitions/AAAAMMJJ_HHMMSS/images"  # dossier (ou motif "*.png") des images à analyser
demi_hauteur = 50      # demi-hauteur (px) de la bande de lignes sommées autour de la raie
lissage = 5            # fenêtre de lissage (px)
plage_nm = (400, 700)  # plage de longueurs d'onde affichée

# ---------------- Calibration en longueur d'onde ----------------
# "aucune"    : axe en pixels
# "theorique" : pente donnée par le modèle (Lambda, f2, ordre, pixel).
#               1 laser de longueur d'onde connue fixe le décalage (sans laser :
#               lbd est supposée centrée sur le capteur).
# "lineaire"  : ajustement pixel -> longueur d'onde sur >= 2 lasers connus
mode_calibration = "theorique"

# Images de chaque laser de calibration (dossier ou motif) + sa longueur d'onde (nm)
lasers_calibration = [
    # {"images": "acquisitions/AAAAMMJJ_HHMMSS/images", "lbd_nm": 532.0},
    # {"images": "acquisitions/AAAAMMJJ_HHMMSS/images", "lbd_nm": 650.0},
]

In [4]:
def wavelength_to_rgb(wavelength, gamma=0.8):

    '''This converts a given wavelength of light to an 
    approximate RGB color value. The wavelength must be given
    in nanometers in the range from 380 nm through 750 nm
    (789 THz through 400 THz).

    Based on code by Dan Bruton
    http://www.physics.sfasu.edu/astro/color/spectra.html
    '''

    wavelength = float(wavelength)
    if wavelength >= 380 and wavelength <= 440:
        attenuation = 0.3 + 0.7 * (wavelength - 380) / (440 - 380)
        R = ((-(wavelength - 440) / (440 - 380)) * attenuation) ** gamma
        G = 0.0
        B = (1.0 * attenuation) ** gamma
    elif wavelength >= 440 and wavelength <= 490:
        R = 0.0
        G = ((wavelength - 440) / (490 - 440)) ** gamma
        B = 1.0
    elif wavelength >= 490 and wavelength <= 510:
        R = 0.0
        G = 1.0
        B = (-(wavelength - 510) / (510 - 490)) ** gamma
    elif wavelength >= 510 and wavelength <= 580:
        R = ((wavelength - 510) / (580 - 510)) ** gamma
        G = 1.0
        B = 0.0
    elif wavelength >= 580 and wavelength <= 645:
        R = 1.0
        G = (-(wavelength - 645) / (645 - 580)) ** gamma
        B = 0.0
    elif wavelength >= 645 and wavelength <= 750:
        attenuation = 0.3 + 0.7 * (750 - wavelength) / (750 - 645)
        R = (1.0 * attenuation) ** gamma
        G = 0.0
        B = 0.0
    else:
        R = 0.0
        G = 0.0
        B = 0.0
    R *= 255
    G *= 255
    B *= 255
    return (int(R), int(G), int(B))

## Fonctions (lecture des images, profil, pic)

In [5]:
def lister_images(chemin):
    """Dossier ou motif glob -> liste triée d'images."""
    if os.path.isdir(chemin):
        fichiers = []
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff"):
            fichiers += glob.glob(os.path.join(chemin, ext))
    else:
        fichiers = glob.glob(chemin)
    fichiers = sorted(fichiers)
    if not fichiers:
        raise FileNotFoundError(f"Aucune image trouvée : {chemin}")
    return fichiers


def lisser(y, n):
    return y if n <= 1 else np.convolve(y, np.ones(n) / n, mode="same")


def image_moyenne(chemin):
    """Moyenne des images (somme des canaux), fond soustrait."""
    fichiers = lister_images(chemin)
    moy = None
    for f in fichiers:
        g = np.asarray(Image.open(f).convert("RGB"), dtype=np.float32).sum(axis=2)
        moy = g if moy is None else moy + g
    moy /= len(fichiers)
    moy -= np.median(moy[::8, ::8])
    np.clip(moy, 0, None, out=moy)
    return moy, len(fichiers)


def profil_intensite(img, demi=None):
    """Intensité en fonction de la colonne (pixel) : moyenne sur une bande de lignes."""
    demi = demi_hauteur if demi is None else demi
    y0 = int(np.argmax(lisser(img.sum(axis=1), 15)))
    y1, y2 = max(0, y0 - demi), min(img.shape[0], y0 + demi)
    return img[y1:y2].mean(axis=0), (y0, y1, y2)


def stats_pic(x, y):
    """Position du max, centroïde (sous-pixel) et FWHM du pic principal."""
    i = int(np.argmax(y))
    demi = y[i] / 2.0
    g = i
    while g > 0 and y[g] > demi:
        g -= 1
    d = i
    while d < len(y) - 1 and y[d] > demi:
        d += 1
    f = slice(g, d + 1)
    return x[i], np.sum(x[f] * y[f]) / np.sum(y[f]), abs(x[d] - x[g])

## Calibration en longueur d'onde (modèle théorique et/ou laser connu)

In [6]:
def pixel_vers_lbd_theorique(px, px_ref, lbd_ref):
    """
    Longueur d'onde (m) à chaque pixel selon le modèle réseau + lentille f2 :
        sin(theta) = ordre*lbd/Lambda + sin(theta_i),   x = f2*tan(theta)
    calé sur un point connu (px_ref, lbd_ref).
    """
    if Lambda is None or f2 is None:
        raise ValueError("Calibration théorique : définir Lambda et f2.")
    th_ref = np.arcsin(ordre * lbd_ref / Lambda + np.sin(theta_i))
    th = np.arctan(np.tan(th_ref) + sens * (px - px_ref) * pixel / f2)
    return Lambda * (np.sin(th) - np.sin(theta_i)) / ordre


def positions_lasers(px):
    """Position (centroïde, pixels) de la raie de chaque laser de calibration."""
    pos, lam = [], []
    for c in lasers_calibration:
        img, n = image_moyenne(c["images"])
        prof, _ = profil_intensite(img)
        _, cent, fwhm = stats_pic(px, lisser(prof, lissage))
        print(f"Laser {c['lbd_nm']:.1f} nm : pixel {cent:.2f} (FWHM {fwhm:.2f} px, {n} images)")
        pos.append(cent)
        lam.append(c["lbd_nm"])
    return np.array(pos), np.array(lam)


def calibrer(n_pixels):
    """Retourne (axe, étiquette, unité) pour les n_pixels du profil."""
    px = np.arange(n_pixels, dtype=float)
    if mode_calibration == "aucune":
        return px, "Position (pixels)", "px"

    pos, lam = positions_lasers(px)

    if mode_calibration == "lineaire":
        if len(pos) < 2:
            raise ValueError("Calibration linéaire : au moins 2 lasers requis.")
        a, b = np.polyfit(pos, lam, 1)
        print(f"Calibration linéaire : lbd = {a:.5f} nm/px * pixel + {b:.3f} nm")
        if len(pos) > 2:
            print("Résidus (nm) :", np.round(lam - (a * pos + b), 3))
        return a * px + b, "Longueur d'onde (nm)", "nm"

    if mode_calibration == "theorique":
        if len(pos) == 0:
            print("ATTENTION : aucun laser -> lbd supposée au centre du capteur.")
            px_ref, lbd_ref = n_pixels / 2, lbd
        else:
            px_ref, lbd_ref = pos[0], lam[0] * 1e-9
            for p, l in zip(pos[1:], lam[1:]):  # vérification du modèle
                l_mod = pixel_vers_lbd_theorique(p, px_ref, lbd_ref) * 1e9
                print(f"Vérification {l:.1f} nm : modèle {l_mod:.2f} nm "
                      f"(écart {l_mod - l:+.2f} nm)")
        x = pixel_vers_lbd_theorique(px, px_ref, lbd_ref) * 1e9
        return x, "Longueur d'onde (nm)", "nm"

    raise ValueError("mode_calibration : 'aucune', 'theorique' ou 'lineaire'.")


def tracer_spectre(ax, x, y, unite):
    """Spectre ; sous la courbe, couleur réelle de chaque longueur d'onde."""
    ax.plot(x, y, "k", lw=0.8)
    if unite == "nm":
        pas = max(1, len(x) // 600)
        for i in range(0, len(x) - pas, pas):
            r, g, b = wavelength_to_rgb(x[i])
            ax.fill_between(x[i:i + pas + 1], y[i:i + pas + 1],
                            color=(r / 255, g / 255, b / 255), lw=0)

## Analyse

In [7]:
# ---------------- Intensité en fonction de la longueur d'onde ----------------
img, n_img = image_moyenne(dossier_mesure)
prof, (y0, y1, y2) = profil_intensite(img)
x, etiquette, unite = calibrer(len(prof))

# Plage affichée et normalisation
masque = (x >= plage_nm[0]) & (x <= plage_nm[1]) if unite == "nm" else np.ones_like(x, bool)
xs = x[masque]
ys = lisser(prof, lissage)[masque]
ys_norm = ys / ys.max()

pos, cent, fwhm = stats_pic(xs, ys)
print(f"{n_img} images | pic : max {pos:.2f} {unite}, centroïde {cent:.2f} {unite}, "
      f"FWHM {fwhm:.2f} {unite}")
if unite == "nm":
    print(f"Pas par pixel : {np.median(np.abs(np.diff(x))):.4f} nm/px")
    if Lambda is not None and L is not None:
        print(f"Résolution théorique (limite d'ouverture) : "
              f"{lbd * Lambda / (ordre * L) * 1e9:.3f} nm")

fig, ax = plt.subplots(2, 1, figsize=(9, 7))
ax[0].imshow(img[::8, ::8], cmap="gray")
ax[0].axhline(y1 / 8, color="r", lw=0.6, ls="--")
ax[0].axhline(y2 / 8, color="r", lw=0.6, ls="--")
ax[0].set_title("Image moyenne et bande d'intégration")
ax[0].axis("off")

tracer_spectre(ax[1], xs, ys_norm, unite)
ax[1].set_xlabel(etiquette)
ax[1].set_ylabel("Intensité normalisée")
ax[1].set_title("Intensité en fonction de la longueur d'onde")
ax[1].grid(alpha=0.3)
fig.tight_layout()
fig.savefig("spectre_intensite.png")
plt.show()

np.savetxt("spectre_intensite.csv",
           np.column_stack([np.arange(len(x))[masque], xs, ys, ys_norm]),
           delimiter=",", header=f"pixel,x_{unite},intensite,intensite_norm", comments="")

FileNotFoundError: Aucune image trouvée : acquisitions/AAAAMMJJ_HHMMSS/images